# Preprocessing

Now that I've explored the data in EDA, I need to get it ready for the models. A few things need to happen:

1. Drop PHQ_total and SEQN — PHQ_total would cause data leakage since the label is built from it, and SEQN is just a patient ID with no predictive value
2. Check for any remaining missing values
3. Encode categorical columns — gender, race, education, marital status are stored as numbers but they're actually categories
4. Scale continuous features — only for Logistic Regression, tree models don't need it
5. Train/test split — 80/20, stratified to keep the same class balance in both sets
6. Save everything to Drive

In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

base = '/content/drive/MyDrive/PTSD_Project'
processed = base + '/data/processed'
results = base + '/results'

print("done")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
done


In [ ]:
#Load data
df = pd.read_csv(processed + '/merged_clean.csv')

print("shape:", df.shape)
print("\nfirst few rows:")
print(df.head())
print("\ncolumns :")
df.columns.tolist()

shape: (5065, 24)

first few rows:
      SEQN  DPQ010  DPQ020  DPQ030  DPQ040  DPQ050  DPQ060  DPQ070  DPQ080  \
0  93705.0       0       0       0       0       0       0       0       0   
1  93706.0       0       0       0       0       0       0       0       0   
2  93708.0       0       0       0       0       0       0       0       0   
3  93711.0       1       0       1       0       0       0       0       0   
4  93712.0       0       0       1       0       0       0       0       0   

   DPQ090  ...  INDFMPIR  DMDMARTL  ALQ111  ALQ121  ALQ130  SLD012  SLQ050  \
0       0  ...      0.82       3.0     1.0     7.0     1.0     8.0     2.0   
1       0  ...      2.13       1.0     2.0     0.0     0.0    10.5     2.0   
2       0  ...      1.63       1.0     2.0     0.0     0.0     8.0     2.0   
3       0  ...      5.00       1.0     1.0     5.0     1.0     7.0     1.0   
4       0  ...      0.76       1.0     1.0     0.0     0.0     7.5     2.0   

   SLQ120  PHQ_total  label

['SEQN',
 'DPQ010',
 'DPQ020',
 'DPQ030',
 'DPQ040',
 'DPQ050',
 'DPQ060',
 'DPQ070',
 'DPQ080',
 'DPQ090',
 'RIAGENDR',
 'RIDAGEYR',
 'RIDRETH1',
 'DMDEDUC2',
 'INDFMPIR',
 'DMDMARTL',
 'ALQ111',
 'ALQ121',
 'ALQ130',
 'SLD012',
 'SLQ050',
 'SLQ120',
 'PHQ_total',
 'label']

## Dropping leaky and irrelevant columns

PHQ_total has to go because the label is directly built from it — if I keep it the model is basically looking at the answer. SEQN is just a row ID, no useful information there.

In [ ]:
df = df.drop(columns=['PHQ_total', 'SEQN'])
print(df.columns.tolist())
print(df.shape)

['DPQ010', 'DPQ020', 'DPQ030', 'DPQ040', 'DPQ050', 'DPQ060', 'DPQ070', 'DPQ080', 'DPQ090', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'DMDEDUC2', 'INDFMPIR', 'DMDMARTL', 'ALQ111', 'ALQ121', 'ALQ130', 'SLD012', 'SLQ050', 'SLQ120', 'label']
(5065, 22)


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5065 entries, 0 to 5064
Data columns (total 22 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   DPQ010    5065 non-null   int64  
 1   DPQ020    5065 non-null   int64  
 2   DPQ030    5065 non-null   int64  
 3   DPQ040    5065 non-null   int64  
 4   DPQ050    5065 non-null   int64  
 5   DPQ060    5065 non-null   int64  
 6   DPQ070    5065 non-null   int64  
 7   DPQ080    5065 non-null   int64  
 8   DPQ090    5065 non-null   int64  
 9   RIAGENDR  5065 non-null   float64
 10  RIDAGEYR  5065 non-null   float64
 11  RIDRETH1  5065 non-null   float64
 12  DMDEDUC2  5065 non-null   float64
 13  INDFMPIR  5065 non-null   float64
 14  DMDMARTL  5065 non-null   float64
 15  ALQ111    5065 non-null   float64
 16  ALQ121    5065 non-null   float64
 17  ALQ130    5065 non-null   float64
 18  SLD012    5065 non-null   float64
 19  SLQ050    5065 non-null   float64
 20  SLQ120    5065 non-null   floa

In [ ]:
df.isnull().sum().sort_values(ascending=False)

,0
DPQ010,0
DPQ020,0
DPQ030,0
DPQ040,0
DPQ050,0
DPQ060,0
DPQ070,0
DPQ080,0
DPQ090,0
RIAGENDR,0


## Encoding categorical columns

Columns like RIAGENDR (gender), RIDRETH1 (race), DMDEDUC2 (education), and DMDMARTL (marital status) look like numbers but they represent categories. Treating them as numbers would be wrong — marital status 3 is not "more than" marital status 1.

Using pd.get_dummies to one-hot encode them. drop_first=True drops one column per feature to avoid multicollinearity.

In [ ]:
cat_cols = ['RIAGENDR', 'RIDRETH1', 'DMDEDUC2', 'DMDMARTL']
df[cat_cols] = df[cat_cols].astype('category')
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(df.shape)
df.columns.tolist()

(5065, 35)


['DPQ010',
 'DPQ020',
 'DPQ030',
 'DPQ040',
 'DPQ050',
 'DPQ060',
 'DPQ070',
 'DPQ080',
 'DPQ090',
 'RIDAGEYR',
 'INDFMPIR',
 'ALQ111',
 'ALQ121',
 'ALQ130',
 'SLD012',
 'SLQ050',
 'SLQ120',
 'label',
 'RIAGENDR_2.0',
 'RIDRETH1_2.0',
 'RIDRETH1_3.0',
 'RIDRETH1_4.0',
 'RIDRETH1_5.0',
 'DMDEDUC2_2.0',
 'DMDEDUC2_3.0',
 'DMDEDUC2_4.0',
 'DMDEDUC2_5.0',
 'DMDEDUC2_7.0',
 'DMDEDUC2_9.0',
 'DMDMARTL_2.0',
 'DMDMARTL_3.0',
 'DMDMARTL_4.0',
 'DMDMARTL_5.0',
 'DMDMARTL_6.0',
 'DMDMARTL_77.0']

After encoding the shape increases because each category value becomes its own binary column. For example RIAGENDR (male/female) becomes one column RIAGENDR_2.0 where 1 means female and 0 means male.

In [ ]:
#train test split
from sklearn.model_selection import train_test_split

X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
X_train.shape, X_test.shape
y_train.value_counts(normalize=True)
y_test.value_counts(normalize=True)

,proportion
label,
0,0.909181
1,0.090819


In [ ]:
cont_cols = ['RIDAGEYR', 'INDFMPIR', 'SLD012']

## Scaling continuous features for Logistic Regression

StandardScaler transforms continuous columns to mean=0, std=1. Logistic Regression is sensitive to feature scales — without scaling, age (18-80) would dominate over income ratio (0-5) just because of its larger range.

Tree-based models don't care about scale so I'm saving two versions of the data.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[cont_cols] = scaler.fit_transform(X_train[cont_cols])
X_test_scaled[cont_cols] = scaler.transform(X_test[cont_cols])

In [ ]:
X_train_scaled[cont_cols].describe()

,RIDAGEYR,INDFMPIR,SLD012
count,4.052000e+03,4.052000e+03,4.052000e+03
mean,-7.101920e-17,1.814935e-16,2.560198e-16
std,1.000123e+00,1.000123e+00,1.000123e+00
min,-1.717751e+00,-1.639952e+00,-3.429877e+00
25%,-8.544213e-01,-8.079765e-01,-3.655016e-01
50%,6.286681e-02,-2.307212e-01,-5.906410e-02
75%,8.182805e-01,7.815431e-01,5.538110e-01
max,1.627652e+00,1.668101e+00,3.924624e+00


In [ ]:
# unscaled (for RF, XGB)
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

train_df.to_csv('/content/drive/MyDrive/PTSD_Project/data/processed/train.csv', index=False)
test_df.to_csv('/content/drive/MyDrive/PTSD_Project/data/processed/test.csv', index=False)

# scaled (for Logistic Regression)
train_scaled_df = pd.concat([X_train_scaled, y_train], axis=1)
test_scaled_df = pd.concat([X_test_scaled, y_test], axis=1)

train_scaled_df.to_csv('/content/drive/MyDrive/PTSD_Project/data/processed/train_scaled.csv', index=False)
test_scaled_df.to_csv('/content/drive/MyDrive/PTSD_Project/data/processed/test_scaled.csv', index=False)